# Section 1 — Inspecting an MCP server

**⏱ About 7 minutes.**

**First, in a terminal:**

```bash
./scripts/run_mcp_server.sh
```

Then run the cells below, top to bottom.

### The idea in one line

We look at the server the way an **agent** does: by *asking* it what it can do,
instead of reading its source. The agent never imports our code.

| Step | We ask | It answers |
|---|---|---|
| 1 | `list_tools()` | what can you do? |
| 2 | read the schema | how do I call it? |
| 3 | `call_tool()` | do it |
| 4 | resources + prompts | what else is there? |
| 5 | raw HTTP | can this scale? |

In [14]:
from fastmcp import Client

# The URL is the entire integration contract. No SDK for our helpdesk, no
# generated client stubs, no shared types. Just an address.
MCP_URL = "http://127.0.0.1:8000/mcp/"

client = Client(MCP_URL)
print("client ready:", MCP_URL)

client ready: http://127.0.0.1:8000/mcp/


## 1. Discovery — what tools exist?

`list_tools()` asks the server to introduce itself. Whatever comes back is
exactly what the model will be offered.

In [15]:
async with client:
    tools = await client.list_tools()

print(f"the server offers {len(tools)} tool(s):\n")

# The description is the first line of the function's docstring. This is the
# text the model reads when deciding which tool to use.
for t in tools:
    print(f"{t.name:24} {t.description.strip().splitlines()[0]}")


the server offers 3 tool(s):

lookup_ticket            Look up a support ticket by its identifier.
search_knowledge_base    Search the internal knowledge base for troubleshooting articles.
open_ticket              Create a new support ticket.


> **What just happened**
>
> We asked an address for a list of capabilities. We wrote no client and no
> shared types — the only thing this notebook knew in advance was a URL.

## 2. The schema — and where it came from

Knowing a tool exists is not enough; the model needs to know *how* to call it.

**We never wrote a JSON schema.** We wrote a Python function, and FastMCP
generated the rest:

```python
@mcp.tool
def lookup_ticket(ticket_id: str) -> dict:
    """Look up a single support ticket by its id..."""
```

The **type hints** became the parameter types. The **docstring** became the
description.

In [12]:
import json

lookup = next(t for t in tools if t.name == "lookup_ticket")

# FastMCP 3.x exposes this MCP wire field as `inputSchema` (camelCase, matching
# the raw JSON). FastMCP 4.x renames it to `input_schema`. We accept either, so
# this cell survives the upgrade -- and so you know the rename is coming.
schema = getattr(lookup, "inputSchema", None) or getattr(lookup, "input_schema")

print(json.dumps(schema, indent=2))


{
  "additionalProperties": false,
  "properties": {
    "ticket_id": {
      "type": "string",
      "description": "The ticket identifier, for example \"TICK-1001\"."
    }
  },
  "required": [
    "ticket_id"
  ],
  "type": "object"
}


> **What just happened**
>
> A JSON schema appeared that nobody wrote by hand.
>
> **The one thing to remember:** that description *is* the prompt the model uses
> to pick this tool. A vague docstring is a bug — most "the agent ignored my
> tool" reports are docstring problems, not code problems.

## 3. Calling a tool

**A tool call is only two things:**

1. a **name** — `"lookup_ticket"`
2. a **dict of arguments** — `{"ticket_id": "TICK-1001"}`

That is the entire thing a model produces when it uses a tool. Below we type
those two things by hand, which is the point: there is nothing else to it.

In [16]:
TOOL_NAME = "lookup_ticket"
ARGUMENTS = {"ticket_id": "TICK-1001"}  # try TICK-1005, or any id in src/helpdesk/data.py

async with client:
    result = await client.call_tool(TOOL_NAME, ARGUMENTS)

print(f"sent:     {TOOL_NAME}{ARGUMENTS}")
print("got back:")

# result.data is the dict the Python function returned, carried back over HTTP.
for key, value in result.data.items():
    print(f"  {key:10} {value}")


sent:     lookup_ticket{'ticket_id': 'TICK-1001'}
got back:
  found      True
  id         TICK-1001
  subject    Laptop will not connect to office VPN
  status     open
  priority   high
  category   network
  requester  dana@example.com
  notes      Fails immediately after MFA prompt. Started after the 14.2 client update.


In [17]:
# A different tool, called exactly the same way: a name, and a dict of arguments.
async with client:
    result = await client.call_tool("search_knowledge_base", {"query": "vpn"})

print(f"{len(result.data)} article(s) matched 'vpn':")
for article in result.data:
    print(f"  {article['id']}  {article['title']}")


1 article(s) matched 'vpn':
  KB-01  Resolving VPN authentication failures


> **What just happened**
>
> Two different tools, called with the same two ingredients: a name and a dict.
>
> **Why it matters:** this request is byte-for-byte what a model sends, so you
> can debug any MCP integration with `curl` and never involve an LLM.
>
> **Try it:** change `ARGUMENTS` to a ticket id that does not exist. The tool
> returns `found: False` instead of raising — a failure the model can recover
> from.

## 4. The other two primitives

MCP has three primitives. They differ by **who decides to use them**:

| Primitive | Who decides | Example here |
|---|---|---|
| Tool | the **model** | `lookup_ticket` |
| Resource | the **application** | `helpdesk://tickets/open` |
| Prompt | the **user** | `triage_prompt` |

In [18]:
# Discovery again -- one list call per primitive, same idea as list_tools().
async with client:
    resources = await client.list_resources()
    prompts = await client.list_prompts()

print("resources:", [str(r.uri) for r in resources])
print("prompts:  ", [p.name for p in prompts])


resources: ['helpdesk://tickets/open']
prompts:   ['triage_prompt']


In [19]:
# A resource is READ by its URI -- no tool name, no arguments, no model involved.
# This text is built from TICKETS in src/helpdesk/data.py, so if you edit that
# file you must restart the server before the change shows up here.
async with client:
    content = await client.read_resource("helpdesk://tickets/open")

print(content[0].text)


6 open ticket(s):
  TICK-1001  [ high ]  Laptop will not connect to office VPN
  TICK-1002  [medium]  Duplicate charge on the September invoice
  TICK-1004  [ high ]  Annual plan renewed at the wrong tier
  TICK-1005  [ high ]  Locked out after too many sign-in attempts
  TICK-1006  [medium]  Docking station no longer charges the laptop
  TICK-1007  [ low  ]  Email client crashes when opening calendar invites


> **What just happened**
>
> We *read* a resource by its URI — no tool name, no arguments, no model
> deciding anything. The application chose to load it.
>
> **Why it matters:** "should this be a tool or a resource?" is a question about
> who decides. Model decides → tool. App always loads it → resource. User picks
> it → prompt.

## 5. Stateless HTTP — the part that matters in production

Our server runs with `stateless_http=True`.

**By default**, MCP hands the client a session id that must be sent on every
later request — so every request in a conversation has to reach **the same
process**. Fine on a laptop; a real problem behind a load balancer.

**The next two cells** drop the SDK and use raw HTTP. First we initialize and
check whether we were given a session id. Then we make a *completely separate*
request, with no session header, and see whether it still works.

> Full explanation: `docs/04-stateless-mcp.md`.

In [21]:
import httpx

# MCP over HTTP is just JSON-RPC 2.0 POSTed to one endpoint.
HEADERS = {
    "Content-Type": "application/json",
    # The server may reply as JSON or as a single SSE event, so accept both.
    "Accept": "application/json, text/event-stream",
}

# The server 307s /mcp/ to /mcp. The FastMCP client handled that for us.
RAW = dict(follow_redirects=True)


def parse(response):
    """Return the JSON-RPC payload, whether it arrived as JSON or SSE."""
    body = response.text
    if body.lstrip().startswith("{"):
        return response.json()
    for line in body.splitlines():
        if line.startswith("data:"):
            return json.loads(line[5:].strip())
    return body


with httpx.Client(**RAW) as http:
    init = http.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {
            "protocolVersion": "2025-06-18",
            "capabilities": {},
            "clientInfo": {"name": "notebook", "version": "1.0"},
        },
    })

print("status:", init.status_code)
# The header that is NOT here is the point.
print("Mcp-Session-Id returned:", init.headers.get("mcp-session-id", "<none>"))

status: 200
Mcp-Session-Id returned: <none>


In [22]:
# A completely separate request. No session header, no shared connection state.
# On a stateful server this would fail with "Missing session ID".
with httpx.Client(**RAW) as http:
    call = http.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"name": "lookup_ticket", "arguments": {"ticket_id": "TICK-1002"}},
    })

print("status:", call.status_code)
print(json.dumps(parse(call), indent=2)[:600])


status: 200
{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "{\"found\":true,\"id\":\"TICK-1002\",\"subject\":\"Duplicate charge on the September invoice\",\"status\":\"open\",\"priority\":\"medium\",\"category\":\"billing\",\"requester\":\"sam@example.com\",\"notes\":\"Two identical line items for the same seat licence.\"}"
      }
    ],
    "structuredContent": {
      "found": true,
      "id": "TICK-1002",
      "subject": "Duplicate charge on the September invoice",
      "status": "open",
      "priority": "medium",
      "category": "


> **What just happened**
>
> The server returned **no** session id, and a second request carrying no
> session still returned a ticket.
>
> **Why it matters:** any instance could have served that request, so this
> scales behind an ordinary load balancer — no sticky sessions, no shared
> session store. That is usually the first question a customer asks.

---

## Recap

| We did | The lesson |
|---|---|
| Asked for tools | the integration contract is a URL, nothing more |
| Read the schema | your docstring is the model's prompt |
| Called a tool | a call is just a name + a dict |
| Read a resource | tool vs resource is about *who decides* |
| Sent raw HTTP | stateless means it scales normally |

Next: `docs/02-lab-a2a.md`.